In [ ]:
'''
In this file, we will work on implementing the Zermelo problem, with some new changes:
- No obstacles
- The state evolution will be deterministic, as it is when one specifies a flight trajectory
- The pseudoloss will aim to encapsulate fuel cost by penalising the cubic of the difference between prescribed and wind velocity
- Of course, it may be difficult to balance this with the cost associated to reaching the destination
- We will set harsh constraints on the maximum and minimum velocities via a sigmoid
- We will model vertical wind via a Brownian bridge from w_min to w_max. No horizontal component just yet

It will be interesting also to investigate whether randomised base policies help (to augment the reference trajectories)
'''

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import math
import pickle
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import HTML
from matplotlib import animation
from matplotlib.collections import LineCollection
import matplotlib.patches as mpatches
import time
import os
from ipywidgets import interact, IntSlider

In [ ]:
# will aim to traverse from (-50, 0) to (50, 0).
# speed of reference trajectories will be 10 units per time step, thus we use time horizon of 10 steps
# will experiment with values for w_min and w_max

def OU_process(x0, sd, theta, mu, sigma, T, time_step):
    total_steps = int(T / time_step)
    process = np.zeros(total_steps + 1, dtype = float)
    process[0] = np.random.normal(loc = x0, scale = sd)
    for t in range(total_steps):
        process[t + 1] = process[t] + theta * (mu - process[t]) * time_step + sigma * np.sqrt(time_step) * np.random.normal()
    return process.astype(np.float32)

def winds(a, b, T, time_step, sigma, theta):
    """
    Simulate OU processes around a and b, then interpolate over the time steps to create an approximate move from a to b over time T, 
    kind of like a wind forecast
    """
    # don't bother with hyperparameters for OU process, just input here
    total_steps = int(T / time_step)
    ou_a = OU_process(a, 0.1, theta, a, sigma, T, time_step)
    ou_b = OU_process(b, 0.1, theta, b, sigma, T, time_step)[::-1]
    interpolants_b = np.linspace(0, 1, total_steps + 1, dtype = np.float32)
    interpolants_a = 1 - interpolants_b
    wind_process = interpolants_a * ou_a + interpolants_b * ou_b
    return wind_process.astype(np.float32)

In [ ]:
for i in range(10):
    plt.plot(winds(-1, 1, 10, 1, 0.4, 0.1))
plt.title("Sample Winds")

In [ ]:
# Device setup
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
torch.set_default_dtype(torch.float32)
print("Running on:", device)

In [ ]:
class NeuralNet_erm(nn.Module):
    def __init__(self, input_dim, width, output_dim):
        super(NeuralNet_erm, self).__init__()
        self.hidden_layer = nn.Linear(input_dim, width)
        self.hidden_layer_2 = nn.Linear(width, width)
        self.hidden_layer_3 = nn.Linear(width, width)
        self.hidden_layer_4 = nn.Linear(width, width)
        self.sigmoid = nn.LeakyReLU()
        self.output_layer = nn.Linear(width, output_dim)
        
    def forward(self, x):
        activations_1 = self.sigmoid(self.hidden_layer(x))
        activations_2 = self.sigmoid(self.hidden_layer_2(activations_1))
        activations_3 = self.sigmoid(self.hidden_layer_3(activations_2))
        activations_4 = self.sigmoid(self.hidden_layer_4(activations_3))
        unscaled = self.output_layer(activations_4)
        return unscaled

# loss function (wind cost + terminal cost)
def running_loss(wind_value, velocity):
    a = velocity.shape[0]
    b = velocity.shape[1]
    zeros_vecs = torch.zeros((a, b, 1), device = device)
    wind_vector = torch.cat([zeros_vecs, wind_value.view(a, b, 1)], dim  = 2)
    wind_cost = torch.norm(wind_vector - velocity, dim = 2)
    return wind_cost

def terminal_loss(position, target):
    ones_vec = torch.ones_like(position, device = device)
    target_matrix = ones_vec * target
    final_cost = torch.norm(position - target_matrix, dim = 1) ** 2
    return final_cost

# def penalising_velocity(velocity, min_speed, max_speed, multiplier, steepness):
#     speed = torch.norm(velocity, dim = 2) # since velocities is 3D tensor
#     min_cost = torch.sigmoid(steepness * (min_speed - speed))
#     max_cost = torch.sigmoid(steepness * (speed - max_speed))
#     return multiplier * torch.relu(min_cost + max_cost - 0.5)

def penalising_velocity(velocity, min_speed, max_speed, multiplier, steepness):
    speed = torch.norm(velocity, dim = 2) # since velocities is 3D tensor
    return multiplier * (torch.relu(speed - max_speed) ** 2 + torch.relu(min_speed - speed) ** 2)

# base policy randomiser function
def random_base_policy(num_policies, num_steps, min_angle, max_angle, base_speed):
    angle_matrix = torch.rand((num_policies, num_steps), device = device) * (max_angle - min_angle) + min_angle
    velocities = torch.stack((torch.cos(angle_matrix), torch.sin(angle_matrix)), dim= -1) * base_speed
    return velocities

pi_tensor = torch.tensor(math.pi, device = device)
two_tensor = torch.tensor(2.0, device = device)
# write the collision-avoidance loss function
def collision_avoidance_loss(positions1, positions2, velocities1, velocities2, T, time_gap, min_distance, collision_multiplier):
    epsilon = 1e-6
    s = min_distance / torch.sqrt(torch.log(two_tensor)) 
    curr_t = T - velocities1.shape[1]
    taus = torch.linspace(0, T, steps = int(T / time_gap) + 1, device = device)[curr_t:-1] # no need to include final time
    dx = positions1 - positions2
    distances = torch.norm(dx, dim = 2) # collapses to (num_policies * n, num_steps)
    distances = torch.clamp(distances, min = 1e-3) # avoid division by zero
    dv = velocities1 - velocities2 
    magvs = torch.norm(dv, dim = 2) 
    magvs = torch.clamp(magvs, min = 1e-3) # avoid division by zero
    dotxv = torch.sum(dx * dv, dim = 2)
    exponent = (dotxv / magvs) ** 2 / 4 - distances
    exponent = torch.clamp(exponent, min=-100, max=100)
    exp_component = torch.exp(exponent / (s ** 2))
    # print(dotxv)
    # print(magvs)
    # print(distances)
    factor = collision_multiplier * ( s * torch.sqrt(pi_tensor)) / (2 * magvs)
    time1 = magvs / s * (taus + dotxv / (2 * magvs ** 2 + epsilon))
    time2 = magvs / s * (taus + dotxv / (2* magvs ** 2 + epsilon) + time_gap)
    collision_costs = factor * exp_component * (torch.erf(time2) - torch.erf(time1))
    return torch.relu(collision_costs - time_gap / 2) # gets rid of penalties from outside radius

def interp_between_columns(x, m, include_last = True):
    samps, ts, d = x.shape
    if ts < 2:
        raise ValueError("T must be >= 2 to interpolate between columns")
    # left and right endpoints for each interval: shape (n, T-1, 1, d)
    left  = x[:, :-1].unsqueeze(2)   # (n, T-1, 1, d)
    right = x[:,  1:].unsqueeze(2)   # (n, T-1, 1, d)
    # alphas: (1, m-1, 1, 1) so it broadcasts over batch and intervals
    alphas = torch.linspace(0.0, 1.0, steps=m, device=x.device, dtype=x.dtype).view(1, 1, m, 1)[:,:,:-1,:]
    # broadcasted interpolation: result -> (n, T-1, m-1, d)
    segs = left * (1-alphas) + right * alphas
    # reorder to put m samples per segment consecutively: (n, T-1, m-1, d) -> (n, (T-1)*(m-1), d)
    segs = segs.reshape(samps, (ts - 1) * (m-1), d)
    if include_last:
        # append final column so final endpoint is present
        last = x[:, -1, :]  # (n,1,d)
        segs = torch.cat([segs, last.view(samps, 1, 2)], dim=1)  # (n, (T-1)*(m-1) + 1, d)
    return segs

def discretised_collision_loss(fineness, pos1, pos2, sep, mult):
    # we need to augment the dx matrix to account for the intermediate collision costs
    # is there an efficient way to perform this interpolation?
    dx = pos1 - pos2
    full_dx = interp_between_columns(dx, fineness)
    distances = torch.norm(dx, dim = 2)
    K = 1 / sep
    return mult * torch.relu(1 / (1 + (K * distances) ** 2)-0.5)


In [ ]:
# visualise trajectories for a set of random base policies
num_policies = 50
num_steps = 10
base_speed = 10.0
policies = random_base_policy(num_policies, num_steps, - 3 * math.pi / 8, math.pi / 8, 11.5)
start = torch.tensor([-50.0, 25.0], device = device)
base_trajectories = torch.zeros((num_policies, num_steps + 1, 2), device = device)
base_trajectories[:, 0, :] = start
# for t in range(num_steps):
#     base_trajectories[:, t + 1, :] = base_trajectories[:, t, :] + policies[:, t, :]
starts = torch.ones((num_policies, 1, 2), device = device) * start
trajectories = torch.cumsum(policies, dim = 1) + torch.tensor([-50, 25], device = device)
trajectories = torch.cat((starts, trajectories), dim = 1)

In [ ]:
plt.plot(trajectories[:,:,0].cpu().numpy().T, trajectories[:,:,1].cpu().numpy().T, linewidth = 0.8)
plt.title("Sample Base Trajectories")
plt.scatter(50, -25, s = 20, color = "black")
plt.show()

In [ ]:
# now, note that our training dataset will be augmented to num_sample * num_policies
def generate_training_data(a, b, T, time_step, n, num_policies, sigma, theta):
    training_data = torch.zeros((n * num_policies, int(T / time_step)), device = device)
    for i in range(n):
        bridge = torch.tensor(winds(a, b, T, time_step, sigma, theta)[: -1], device = device)
        bridge = bridge.repeat(num_policies, 1)
        training_data[i * num_policies : (i + 1) * num_policies, :] = bridge
    return training_data

In [ ]:
generate_training_data(-1, 1, 10, 1, 5, 3, 0.2, 0.1)

In [ ]:
# hyperparameters
input_dim = 3
width = 500
output_dim = 2
v_min = 7
v_max = 23
velocity_penalty_multiplier = 1000
n = 20
num_policies = 60
T = 10
time_step = 1
num_steps = int(T / time_step)
base_speed = 11.3
a = -1
b = 1
start_rate = 0.0001
final_rate = 0.0000000001
num_epochs = 2000
target = torch.tensor([50.0, -25.0], device = device)
steepness = 5
sigma = 0.2
theta = 0.1
outer_loops = 8

In [ ]:
Zs = np.linspace(-2, 2, 20)

# Create grid
xs = np.linspace(-50, 50, 50)
ys = np.linspace(-50, 50, 50)
X, Y = np.meshgrid(xs, ys)

def plot_quiver(model_idx, z_idx):
    model = models_erm[model_idx]
    z = Zs[z_idx]
    xy = np.stack([X.ravel(), Y.ravel(), np.full_like(X.ravel(), z)], axis=1)
    xy_tensor = torch.tensor(xy, dtype=torch.float32, device = device)
    with torch.no_grad():
        output = model(xy_tensor).cpu().numpy()

    U = output[:,0].reshape(X.shape)
    V = output[:,1].reshape(X.shape)

    # Plot quiver
    plt.figure(figsize=(10,5))
    plt.quiver(X, Y, U, V, scale = 1000)

    for pth in range(n):
        plt.plot(current_paths[pth * num_policies, :model_idx + 1, 0].detach().cpu().numpy(), 
                 current_paths[pth * num_policies, :model_idx + 1, 1].detach().cpu().numpy(), color = 'red', linewidth = 0.5, alpha = 0.9)
        plt.plot(current_paths[pth * num_policies, model_idx:, 0].detach().cpu().numpy(), 
                 current_paths[pth * num_policies, model_idx:, 1].detach().cpu().numpy(), color = 'blue', linewidth = 0.5, alpha = 0.9)
        
    plt.title(f"PyTorch Quiver Plot | Model {model_idx+1} | Z = {z}")
    plt.xlabel("X")
    plt.ylabel("Y")
    plt.xlim(-50, 50)
    plt.ylim(-50, 50)
    plt.grid(True)
    plt.show()

In [ ]:
''' New idea:
Since we are modelling deterministically, instead of penalising the terminal cost as squared distance to target, 
we can just set it to be the required remaining velocity to hit the target perfectly (a deterministic object!), 
and penalise the resulting velocity decision accordingly (which is a function of T-1 th model)
'''
# generate the training data
training_data = generate_training_data(a, b, T, time_step, n, num_policies, sigma, theta)

# will also need the reference trajectories (same set of num_policies for each sample)
ref_paths = torch.zeros((n * num_policies, num_steps + 1, 2), device = device)
starts = torch.ones((num_policies, 1, 2), device = device) * torch.tensor([-50, 25], device = device)
for i in range(n):
    base_policies = random_base_policy(num_policies, num_steps, - 3 * math.pi / 8, math.pi / 8, base_speed)
    starts = torch.ones((num_policies, 1, 2), device = device) * torch.tensor([-50, 25], device = device)
    ref_pathsi = torch.cumsum(base_policies, dim = 1) + torch.tensor([-50, 25], device = device)
    ref_pathsi = torch.cat((starts, ref_pathsi), dim = 1)
    # repeat for each instance of the training samples
    # ref_paths = ref_paths.repeat(n, 1, 1)
    ref_paths[i * num_policies : (i + 1) * num_policies, :, :] = ref_pathsi
# training data and reference paths now fully constructed
# initialise models, optimisers and schedulers
models_erm = [NeuralNet_erm(input_dim, width, output_dim).to(device) for _ in range(T - 1)] # final velocity is a function of prior one now
optimisers = [optim.AdamW(model.parameters(), lr=start_rate) for model in models_erm]
schedulers = [optim.lr_scheduler.CosineAnnealingLR(opt, T_max = num_epochs, eta_min = final_rate) for opt in optimisers]

In [ ]:
for loop in range(outer_loops):
    print(f"Outer loop {loop + 1}")
    for t in reversed(range(T - 1)):
        print(f"t = {t}")
        path_length = T - t
        for epoch in range(num_epochs):
            final_c = 0
            current_paths = [ref_paths[:, t, :]]
            velocities = []
            # generate path from t
            for futs in range(path_length - 1): # up until T-1 then final velocity calculated separately
                velocity = models_erm[t + futs](torch.cat([current_paths[-1],
                                                        training_data[:, t + futs].view(n * num_policies, 1)], 
                                                        dim = 1))
                current_paths.append(current_paths[-1] + time_step * velocity)
                velocities.append(velocity)
            # final velocity to hit target exactly
            final_velocity = (target - current_paths[-1]) / time_step
            velocities.append(final_velocity)
            # stack into 3D tensors
            current_paths = torch.stack(current_paths, dim = 1)
            to_add = target.repeat(n * num_policies, 1).view(n * num_policies, 1, 2)
            current_paths = torch.cat([current_paths, to_add], dim = 1)
            velocities = torch.stack(velocities, dim = 1)
            # compute losses
            safety_costs = penalising_velocity(velocities, v_min, v_max, velocity_penalty_multiplier, steepness)
            safety_costs = torch.sum(safety_costs, dim = 1)
            running_losses = running_loss(training_data[:, t:], velocities)
            running_costs = torch.sum(running_losses, dim = 1) / (T)  # kind of normalises
            final_c = final_c + torch.mean(safety_costs + running_costs)
            # backprop
            final_c.backward()
            optimisers[t].step()
            optimisers[t].zero_grad()
            schedulers[t].step()
            if epoch % 1000 == 0:
                with torch.no_grad():
                    print(f"Epoch {epoch}, Total {final_c:.6f}, Safety {torch.mean(safety_costs):.6f}, Running {torch.mean(running_costs):.6f}")
        # freeze model after training
        for param in models_erm[t].parameters():
            param.requires_grad = False
    for model in models_erm:
        for param in model.parameters():
            param.requires_grad = True
    # reset optimisers and schedulers
    optimisers = [optim.AdamW(model.parameters(), lr=start_rate) for model in models_erm]
    schedulers = [optim.lr_scheduler.CosineAnnealingLR(opt, T_max = num_epochs, eta_min = final_rate) for opt in optimisers]
    # new reference policies are the learned policies for this loop
    ref_paths = current_paths.detach()

In [ ]:
for pth in range(n):
    plt.plot(current_paths[pth * num_policies, :, 0].detach().cpu().numpy(), 
             current_paths[pth * num_policies, :, 1].detach().cpu().numpy())
plt.title("Learned Trajectories")

In [ ]:
interact(
    plot_quiver,
    model_idx=IntSlider(min=0, max=len(models_erm)-1, step=1, description='Model'),
    z_idx=IntSlider(min=0, max=len(Zs)-1, step=1, description='Z index')
)
plt.show()

In [ ]:
magvs = torch.norm(velocities,dim =2)
import seaborn as sns
sns.heatmap(magvs.detach().cpu().numpy())

In [ ]:
'''
Now we extend to two planes, by just adding another plane on top of this problem.

Same start time, end time, and wind conditions, but different start and target positions.


'''


pi_tensor = torch.tensor(math.pi, device = device)
two_tensor = torch.tensor(2.0, device = device)

In [ ]:
# hyperparameters
input_dim = 3
width = 500
output_dim = 2
v_min = 7
v_max = 23
velocity_penalty_multiplier = 1000
collision_multiplier = 10000
n = 20
num_policies = 60
T = 10
time_step = 1
num_steps = int(T / time_step)
base_speed = 11.3
a = -1
b = 1
start_rate = 0.0001
final_rate = 0.000001
num_epochs = 4000
steepness = 5
sigma = 0.2
theta = 0.1
outer_loops = 3
fineness_param = 5
sep = 10
col_mult = 10
# training data stays the same
start2 = torch.tensor([-50.0, -25.0], device = device)
target2 = torch.tensor([50.0, 25.0], device = device)
# will also need the reference trajectories (same set of num_policies for each sample)
base_policies2 = random_base_policy(num_policies, num_steps, -math.pi/8, 3*math.pi/8, base_speed)
starts2 = torch.ones((num_policies, 1, 2), device = device) * start2
ref_paths2 = torch.cumsum(base_policies2, dim = 1) + start2
ref_paths2 = torch.cat((starts2, ref_paths2), dim = 1)
# repeat for each instance of the training samples
ref_paths2 = ref_paths2.repeat(n, 1, 1)
# reference paths now fully constructed
# initialise models, optimisers and schedulers
models_erm2 = [NeuralNet_erm(input_dim, width, output_dim).to(device) for _ in range(T - 1)] # final velocity is a function of prior one now
optimisers2 = [optim.AdamW(model.parameters(), lr=start_rate) for model in models_erm2]
schedulers2 = [optim.lr_scheduler.CosineAnnealingLR(opt, T_max = num_epochs, eta_min = final_rate) for opt in optimisers2]

In [ ]:
# first models stay frozen, train second models
for model in models_erm:
    for parameter in model.parameters():
        parameter.requires_grad = False

for loop in range(outer_loops):
    print(f"Outer loop {loop}")
    for t in reversed(range(T - 1)):
        print(f"t = {t}")
        path_length = T - t
        for epoch in range(num_epochs):
            final_c2 = 0
            current_paths2 = [ref_paths2[:, t, :]]
            velocities2 = []
            # generate path from t
            for futs in range(path_length - 1): # up until T-1 then final velocity calculated separately
                velocity = models_erm2[t + futs](torch.cat([current_paths2[-1],
                                                        training_data[:, t + futs].view(n * num_policies, 1)], 
                                                        dim = 1))
                current_paths2.append(current_paths2[-1] + time_step * velocity)
                velocities2.append(velocity)
            # final velocity to hit target exactly
            final_velocity = (target2 - current_paths2[-1]) / time_step
            velocities2.append(final_velocity)
            # stack into 3D tensors
            current_paths2 = torch.stack(current_paths2, dim = 1) # shape is (n * num_policies, path_length, 2)
            to_add = target2.repeat(n * num_policies, 1).view(n * num_policies, 1, 2)
            current_paths2 = torch.cat([current_paths2, to_add], dim = 1)
            velocities2 = torch.stack(velocities2, dim = 1)
            # compute losses
            safety_costs2 = penalising_velocity(velocities2, v_min, v_max, velocity_penalty_multiplier, steepness)
            safety_costs = torch.sum(safety_costs2, dim = 1)
            running_losses2 = running_loss(training_data[:, t:], velocities2)
            running_costs = torch.sum(running_losses2, dim = 1) / T # kind of normalises
            # collision_costs = collision_avoidance_loss(current_paths[:, t:-1, :].detach(), 
            #                                            current_paths2[:, :-1, :], 
            #                                            velocities[:, t:, :].detach(),
            #                                            velocities2,
            #                                            T,
            #                                            time_gap = time_step,
            #                                            min_distance = min_distance,
            #                                            collision_multiplier = collision_multiplier)
            collision_costs = discretised_collision_loss(fineness_param, 
                                                         current_paths[:, t:-1, :].detach(),
                                                         current_paths2[:, :-1, :],
                                                         sep, 
                                                         col_mult)
            collision_costs = torch.sum(collision_costs, dim = 1)
            final_c2 = final_c2 + torch.mean(safety_costs + running_costs + collision_costs)
            # backprop
            final_c2.backward()
            optimisers2[t].step()
            optimisers2[t].zero_grad()
            schedulers2[t].step()
            if epoch % 1000 == 0:
                with torch.no_grad():
                    print(f"Epoch {epoch}, Total {final_c2:.6f}, Safety {torch.mean(safety_costs):.6f}, Running {torch.mean(running_costs):.6f},Collision {torch.mean(collision_costs):.6f}")
        # freeze model after training
        for param in models_erm2[t].parameters():
            param.requires_grad = False
    for model in models_erm2:
        for param in model.parameters():
            param.requires_grad = True
    # reset optimisers and schedulers
    optimisers2 = [optim.AdamW(model.parameters(), lr=start_rate) for model in models_erm2]
    schedulers2 = [optim.lr_scheduler.CosineAnnealingLR(opt, T_max = num_epochs, eta_min = final_rate) for opt in optimisers2]
    # new reference policies are the learned policies for this loop
    ref_paths2 = current_paths2.detach()

In [ ]:
for pth in range(n):
    plt.plot(current_paths[pth * num_policies, :, 0].detach().cpu().numpy(), 
             current_paths[pth * num_policies, :, 1].detach().cpu().numpy(), color = 'red')
    plt.plot(current_paths2[pth * num_policies, :, 0].detach().cpu().numpy(), 
             current_paths2[pth * num_policies, :, 1].detach().cpu().numpy(), color = 'blue')
plt.title("Learned Trajectories for Two Planes")

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

def interpolate_trajectory(positions, n_interp=10):
    """
    Linearly interpolate positions between timesteps to make animation smoother.
    positions: [T, 4] (single sample)
    Returns: [T * n_interp, 4]
    """
    T = positions.shape[0]
    interp_positions = []
    for t in range(T - 1):
        start = positions[t]
        end = positions[t + 1]
        for i in range(n_interp):
            alpha = i / n_interp
            interp = (1 - alpha) * start + alpha * end
            interp_positions.append(interp)
    interp_positions.append(positions[-1])  # include last timestep
    return np.stack(interp_positions, axis=0)

def animate_smooth_positions(current_paths, current_paths2, sample_idx=0, interval=50, n_interp=10):
    pos1 = current_paths[sample_idx].detach().cpu().numpy()  # [T, 2]
    smooth_pos1 = interpolate_trajectory(pos1, n_interp=n_interp)
    
    pos2 = current_paths2[sample_idx].detach().cpu().numpy()  # [T, 2]
    smooth_pos2 = interpolate_trajectory(pos2, n_interp=n_interp)

    x1, y1 = smooth_pos1[:, 0], smooth_pos1[:, 1]
    x2, y2 = smooth_pos2[:, 0], smooth_pos2[:, 1]

    fig, ax = plt.subplots(figsize=(6,6))
    ax.set_xlim(-50, 50)
    ax.set_ylim(-30, 30)

    line1, = ax.plot([], [], 'b-', label='Plane 1')
    line2, = ax.plot([], [], 'r-', label='Plane 2')
    point1, = ax.plot([], [], 'bo')
    point2, = ax.plot([], [], 'ro')
    ax.legend()

    def init():
        line1.set_data([], [])
        line2.set_data([], [])
        point1.set_data([], [])
        point2.set_data([], [])
        return line1, line2, point1, point2

    def update(frame):
        line1.set_data(x1[:frame+1], y1[:frame+1])
        line2.set_data(x2[:frame+1], y2[:frame+1])
        point1.set_data([x1[frame]], [y1[frame]])
        point2.set_data([x2[frame]], [y2[frame]])
        return line1, line2, point1, point2

    ani = FuncAnimation(fig, update, frames=len(x1), init_func=init, interval=interval, blit=False)
    return HTML(ani.to_jshtml())




In [ ]:
animate_smooth_positions(current_paths, current_paths2, sample_idx=3, interval=150, n_interp=10)

In [ ]:
for model in models_erm:
    for parameter in model.parameters():
        parameter.requires_grad = False

# what about if we just one very large neural network for it all?
input_dim = 4
width = 2000 # a bit wider as must learn more complex behaviour
num_epochs = 20000
start_rate = 0.001
final_rate = 0.0000001
full_erm = NeuralNet_erm(input_dim, width, output_dim).to(device)
optimisers_full = optim.AdamW(full_erm.parameters(), lr=start_rate)
schedulers_full = optim.lr_scheduler.CosineAnnealingLR(optimisers_full, T_max = num_epochs, eta_min = final_rate)
starts3 = torch.ones((n * num_policies, 2), device = device) * start2
target3 = target2
for epoch in range(num_epochs):
    # reinitialise current paths
    current_paths3 = [starts3]
    velocities3 = []
    final_c3 = 0.0
    for t in range(T-1): # final velocity decided deterministically
        velocity3 = full_erm(torch.cat([current_paths3[-1], 
                                        training_data[:, t].view(n * num_policies, 1), 
                                        t * torch.ones(n * num_policies, 1, 
                                                       device = device)], 
                                                       dim = 1))
        current_paths3.append(current_paths3[-1] + time_step * velocity3)
        velocities3.append(velocity3)
    # append final velocity and target
    final_v = (target3 - current_paths3[-1]) / time_step
    current_paths3.append(target3 * torch.ones(n * num_policies, 2, device = device))
    velocities3.append(final_v)
    # stack lists into tensors
    current_paths3 = torch.stack(current_paths3, dim = 1)
    velocities3 = torch.stack(velocities3, dim = 1)
    # compute losses
    safety_costs3 = penalising_velocity(velocities3, v_min, v_max, velocity_penalty_multiplier, steepness)
    safety_costs3 = torch.sum(safety_costs3, dim = 1)
    running_losses3 = running_loss(training_data, velocities3)
    running_costs3 = torch.sum(running_losses3, dim = 1) / T # kind of normalises
    # collision_costs = collision_avoidance_loss(current_paths[:, t:-1, :].detach(), 
    #                                            current_paths2[:, :-1, :], 
    #                                            velocities[:, t:, :].detach(),
    #                                            velocities2,
    #                                            T,
    #                                            time_gap = time_step,
    #                                            min_distance = min_distance,
    #                                            collision_multiplier = collision_multiplier)
    collision_costs3 = discretised_collision_loss(fineness_param, 
                                                    current_paths[:, :-1, :].detach(),
                                                    current_paths3[:, :-1, :],
                                                    sep, 
                                                    col_mult)
    collision_costs3 = torch.sum(collision_costs3, dim = 1)
    final_c3 = final_c3 + torch.mean(safety_costs3 + running_costs3 + collision_costs3)
    # backprop
    optimisers_full.zero_grad()
    final_c3.backward()
    optimisers_full.step()
    
    schedulers_full.step()
    if epoch % 100 == 0:
        with torch.no_grad():
            print(f"Epoch {epoch}, Total {final_c3:.6f}, Safety {torch.mean(safety_costs3):.6f}, Running {torch.mean(running_costs3):.6f},Collision {torch.mean(collision_costs3):.6f}")

In [ ]:
animate_smooth_positions(current_paths, current_paths3, sample_idx=3, interval=150, n_interp=10)